# M3 Agentic AI - 代码执行 Code Execution

## 1. 引言

### 1.1 实验概览

代码执行（Code execution）是让大型语言模型（LLM）能够编写并运行代码，以解决用户提出的任务。

### 1.2 核心优势
- **强大能力**: LLM生成的代码方案常常出人意料地聪明和有效，能解决各种复杂任务。
- **灵活性**: 相比于为每个功能手动创建独立工具，让模型自己写代码是一种更灵活、可扩展性更强的方法。
- **提升应用性能**: 许多语言模型的训练者会专门优化模型，以确保其在应用中代码执行功能表现良好。

### 🎯 1.3 学习目标
在本实验结束时，你将能够：
- 理解代码执行相比预定义工具的优势
- 实现一个让模型自己编写并执行代码的智能体系统
- 应用反思模式处理代码执行错误
- 理解安全代码执行的重要性

## 2. 初始化环境与客户端

导入必要的库并初始化客户端。

In [1]:
import sys
sys.path.append("../")
import json
import re
import math
from typing import Dict, Any, Optional
from utils import *
# import display_functions

# 使用 OpenAI 接口连接到 Ollama
from openai import OpenAI
client = OpenAI(
    base_url="http://127.0.0.1:8080",
    api_key="sk-no-key-required"
)
qwen_model = 'qwen3.5-35b-a3b-q4km'

## 3. 传统方法 vs 代码执行

### 3.1 场景：简单计算器

**传统方法：预定义工具集**

1. **场景**：构建一个能输入数学文字题并解答的应用。
2. **方法**：创建一系列预定义工具（如 `add`, `subtract`, `multiply`, `divide`）。
3. **流程**：
   - 用户输入："Add 13.2 and 18.9"。
   - LLM 识别需求，调用 `add` 工具。
   - 工具执行计算，返回结果 `32.1`。
   - LLM 将结果格式化后返回给用户。
4. **局限性**：
   - 面对新需求（如"2的平方根是多少？"），需要不断添加新工具（如 `sqrt`, `exponentiation`）。
   - 现代科学计算器功能繁多，为每个按钮都创建独立工具不切实际。

**替代方案：让模型自己写代码**

- **核心理念**：与其逐个实现功能，不如让系统自己编写并执行代码来解决问题。
- **优势**：极大地扩展了模型的能力边界，使其能处理任何可以通过编程解决的问题。

## 4. 实现代码执行智能体

### 4.1 系统提示词设计

设计系统提示词，指令模型：
- 编写代码来解决用户的问题
- 将答案以 Python 代码形式返回
- 用 `<execute_python>` 和 `</execute_python>` 标签包裹代码

In [2]:
def build_code_prompt(request_: str) -> str:
    """构建代码执行智能体的提示词"""
    return f"""
- 你是一名专注于代码执行的 AI 助理。
- 对于用户的数学或计算问题，编写 Python 代码来解决它。
- 将你的代码用 <execute_python> 和 </execute_python> 标签包裹。
- 代码应该使用 print() 函数输出最终结果。
- 不要解释代码，直接返回包裹的代码。

{request_.strip()}
"""

### 4.2 代码提取器

使用正则表达式从模型输出中提取被标签包裹的代码。

In [3]:
def extract_python_code(text: str) -> Optional[str]:
    """从文本中提取被 <execute_python> 和 </execute_python> 包裹的代码"""
    pattern = r'<execute_python>(.*?)</execute_python>'
    match = re.search(pattern, text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return None

# 测试代码提取
test_text = '''
这是代码：<execute_python>
import math
print(math.sqrt(2))
</execute_python>
'''

code = extract_python_code(test_text)
print("提取的代码:")
print(code)

提取的代码:
import math
print(math.sqrt(2))


### 4.3 代码执行器

使用 Python 的 `exec()` 函数执行提取出的代码。

⚠️ **注意**：在生产环境中应该使用安全的沙盒环境（如 Docker 或 E2B）。

In [4]:
import io
import sys
from contextlib import redirect_stdout

def execute_python_code(code: str) -> Dict[str, Any]:
    """安全地执行 Python 代码并返回输出"""
    # 重定向标准输出
    output_buffer = io.StringIO()
    
    result = {
        "success": False,
        "output": "",
        "error": None
    }
    
    try:
        # 创建执行环境
        exec_globals = {
            '__builtins__': {
                'print': print,
                'len': len,
                'range': range,
                'list': list,
                'dict': dict,
                'str': str,
                'int': int,
                'float': float,
                'bool': bool,
                'abs': abs,
                'min': min,
                'max': max,
                'sum': sum,
                'round': round,
                # 添加数学库
                'math': math,
                '__import__': __import__,
            }
        }
        
        # 执行代码并捕获输出
        with redirect_stdout(output_buffer):
            exec(code, exec_globals, {})
        
        output = output_buffer.getvalue().strip()
        result["success"] = True
        result["output"] = output
        
    except Exception as e:
        result["error"] = str(type(e).__name__) + ": " + str(e)
        result["output"] = output_buffer.getvalue().strip()
    
    return result

# 测试代码执行
test_code = '''
import math
result = math.sqrt(2)
print(f"sqrt(2) = {result}")
'''

exec_result = execute_python_code(test_code)
print("执行结果:")
print(json.dumps(exec_result, indent=2, ensure_ascii=False))

执行结果:
{
  "success": true,
  "output": "sqrt(2) = 1.4142135623730951",
  "error": null
}


### 4.4 完整的代码执行智能体

In [5]:
def code_execution_agent(prompt: str, max_iterations: int = 3):
    """代码执行智能体
    
    Args:
        prompt: 用户的问题或请求
        max_iterations: 最大迭代次数（用于反思模式）
    
    Returns:
        包含执行结果和对话历史的字典
    """
    messages = [{"role": "user", "content": build_code_prompt(prompt)}]
    
    for iteration in range(max_iterations):
        # 调用 LLM 生成代码
        response = client.chat.completions.create(
            model=qwen_model,
            messages=messages
        )
        
        # 提取代码
        code = extract_python_code(response.choices[0].message.content)
        
        if code is None:
            # 没有检测到代码，直接返回
            return {
                "response": response,
                "messages": messages,
                "execution_result": None,
                "iterations": iteration
            }
        
        # 执行代码
        exec_result = execute_python_code(code)
        
        # 如果成功，格式化结果并返回
        if exec_result["success"]:
            # 让 LLM 格式化结果
            format_prompt = f"""
                            用户的问题: {prompt}

                            代码执行结果: {exec_result['output']}

                            请根据用户的问题，将代码执行结果格式化为一个友好、清晰的答案。
                            """
            
            messages.append({"role": "assistant", "content": response.choices[0].message.content})
            messages.append({"role": "user", "content": format_prompt})
            
            final_response = client.chat.completions.create(
                model=qwen_model,
                messages=messages
            )
            
            return {
                "response": final_response,
                "messages": messages + [final_response.choices[0].message],
                "execution_result": exec_result,
                "code": code,
                "iterations": iteration + 1
            }
        
        # 如果失败且还有迭代机会，将错误信息反馈给 LLM（反思模式）
        if iteration < max_iterations - 1:
            error_feedback = f"""
                            代码执行失败！错误信息:
                            {exec_result['error']}

                            执行的代码:
                            {code}

                            请修复代码中的错误并重新生成。
                            """
            
            messages.append({"role": "assistant", "content": response.choices[0].message.content})
            messages.append({"role": "user", "content": error_feedback})
        else:
            # 达到最大迭代次数，返回错误
            return {
                "response": response,
                "messages": messages,
                "execution_result": exec_result,
                "code": code,
                "iterations": iteration + 1
            }
    
    return None

## 5. 测试代码执行智能体

### 5.1 简单数学计算

In [6]:
# 测试 1: 简单计算
result = code_execution_agent("What's the square root of 2?")

print("=" * 50)
print("问题: What's the square root of 2?")
print("=" * 50)
print("\n生成的代码:")
print(result.get("code", "N/A"))
print("\n代码执行结果:")
print(json.dumps(result.get("execution_result"), indent=2, ensure_ascii=False))
print("\n最终答案:")
print(result["response"].choices[0].message.content)
print(f"\n迭代次数: {result.get('iterations', 0)}")

问题: What's the square root of 2?

生成的代码:
import math
print(math.sqrt(2))

代码执行结果:
{
  "success": true,
  "output": "1.4142135623730951",
  "error": null
}

最终答案:
2 的平方根约为 **1.4142135623730951**。

这是一个无理数，通常可以近似为 **1.414**（保留三位小数）或 **1.41**（保留两位小数）。

迭代次数: 1


### 5.2 复杂计算

In [7]:
# 测试 2: 复杂计算
result = code_execution_agent("Calculate the area of a circle with radius 5.")

print("=" * 50)
print("问题: Calculate the area of a circle with radius 5")
print("=" * 50)
print("\n生成的代码:")
print(result.get("code", "N/A"))
print("\n代码执行结果:")
print(json.dumps(result.get("execution_result"), indent=2, ensure_ascii=False))
print("\n最终答案:")
print(result["response"].choices[0].message.content)

问题: Calculate the area of a circle with radius 5

生成的代码:
import math

radius = 5
area = math.pi * radius ** 2
print(area)

代码执行结果:
{
  "success": true,
  "output": "78.53981633974483",
  "error": null
}

最终答案:
The area of a circle with a radius of 5 is approximately 78.54.


### 5.3 批量计算

In [8]:
# 测试 3: 批量计算
result = code_execution_agent("Find the sum of squares of numbers from 1 to 10.")

print("=" * 50)
print("问题: Find the sum of squares of numbers from 1 to 10")
print("=" * 50)
print("\n生成的代码:")
print(result.get("code", "N/A"))
print("\n代码执行结果:")
print(json.dumps(result.get("execution_result"), indent=2, ensure_ascii=False))
print("\n最终答案:")
print(result["response"].choices[0].message.content)

问题: Find the sum of squares of numbers from 1 to 10

生成的代码:
print(sum(i**2 for i in range(1, 11)))

代码执行结果:
{
  "success": true,
  "output": "385",
  "error": null
}

最终答案:
从 1 到 10 的数字的平方和是 **385**。

具体计算如下：
- 1² = 1
- 2² = 4
- 3² = 9
- 4² = 16
- 5² = 25
- 6² = 36
- 7² = 49
- 8² = 64
- 9² = 81
- 10² = 100

**总计：1 + 4 + 9 + 16 + 25 + 36 + 49 + 64 + 81 + 100 = 385**


## 6. 进阶：带外部反馈的反思（Reflection with External Feedback）

### 6.1 概念

- **反思模式**：当代码执行失败时，将错误信息（如 `SyntaxError`）反馈给模型。
- **流程**：
  1. 模型生成第一版代码草稿。
  2. 执行代码，捕获错误。
  3. 将错误信息连同原始任务一起发送给模型。
  4. 模型反思错误，生成改进后的第二版代码。
- **效果**：通过这种迭代过程，模型有时能修正错误，得到更精确的答案。

### 6.2 测试反思模式

让我们故意生成一个可能有错误的场景，观察模型如何自我修正。

In [12]:
# 测试反思模式 - 模拟一个可能导致错误的复杂问题
result = code_execution_agent(
    "Calculate 100 factorial (100!) and tell me how many digits it has.",
    max_iterations=3
)

print("=" * 50)
print("问题: Calculate 100 factorial and tell me how many digits it has")
print("=" * 50)
print("\n生成的代码:")
print(result.get("code", "N/A"))
print("\n代码执行结果:")
exec_result = result.get("execution_result")
if exec_result:
    print(json.dumps(exec_result, indent=2, ensure_ascii=False))
print("\n最终答案:")
print(result["response"].choices[0].message.content)
print(f"\n迭代次数: {result.get('iterations', 0)}")

问题: Calculate 100 factorial and tell me how many digits it has

生成的代码:
import math
factorial_100 = math.factorial(100)
print(len(str(factorial_100)))

代码执行结果:
{
  "success": true,
  "output": "158",
  "error": null
}

最终答案:
100 的阶乘（100!）是一个非常大的数，共有 **158 位数字**。具体数值太大无法直接展示，但通过计算可确认其位数。

迭代次数: 1


## 7. 安全考量：安全的代码执行（Secure Code Execution）

### 7.1 核心风险

- **在沙盒环境外运行由模型生成的任意代码是有风险的**。
- **真实案例**：
  - 一位团队成员使用的高度智能体化代码执行器，曾错误地执行了 `rm *.py` 命令，删除了项目目录中的所有 Python 文件。
  - 幸运的是，该成员有备份习惯，未造成实质损失。

### 7.2 最佳实践

- **必须使用沙盒环境**：这是保护系统、防止数据丢失或敏感数据泄露的最佳做法。
- **轻量级沙盒**：推荐使用像 Docker 或 E2B 这样的轻量级沙盒环境，它们能有效隔离代码，降低损坏系统或环境的风险。
- **限制可用函数**：如我们在 `execute_python_code` 中所做的那样，只暴露安全的内置函数。

### 7.3 当前实现的限制

我们的当前实现做了一些安全限制：
- 只暴露了部分安全的 Python 内置函数
- 允许访问 `math` 模块
- **没有**暴露文件操作、网络操作等危险功能

但请注意，这个实现仍然不够安全，生产环境应该使用专门的沙盒环境。

## 8. 对比：预定义工具 vs 代码执行

### 8.1 预定义工具方法

**优点**：
- 安全性高（完全控制可执行的操作）
- 行为可预测
- 性能稳定

**缺点**：
- 需要预先实现所有可能的功能
- 扩展性差（新功能需要开发）
- 对于复杂计算场景，需要大量工具

**示例代码**：
```python
# 需要为每个操作定义工具
def add(a, b):
    return a + b

def subtract(a, b):
    return a - b

def sqrt(x):
    return math.sqrt(x)

# ... 需要为每个功能添加工具
```

### 8.2 代码执行方法

**优点**：
- 极大的灵活性
- 无需预定义工具
- 可以处理任何可编程解决的问题
- 扩展性强

**缺点**：
- 安全性风险（需要沙盒环境）
- 执行性能可能不稳定
- 错误处理更复杂

**示例代码**：
```python
# 一个通用的代码执行工具
def execute_code(code):
    return execute_python_code(code)
```

## 9. 实际应用场景

### 9.1 科学计算

代码执行非常适合科学计算场景：
- 数学问题求解
- 数据分析
- 统计计算
- 算法实现

In [14]:
# 实际应用：数学问题
result = code_execution_agent(
    "Find all prime numbers less than 50."
)

print("=" * 50)
print("问题: Find all prime numbers less than 50")
print("=" * 50)
print("\n最终答案:")
print(result["response"].choices[0].message.content)

问题: Find all prime numbers less than 50

最终答案:
以下是小于 50 的所有质数：

2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47

这些数除了 1 和自身外，没有其他正因数。


### 9.2 数据处理

代码执行可以处理复杂的数据处理任务。

In [15]:
# 实际应用：数据处理
result = code_execution_agent(
    "I have these numbers: [23, 45, 12, 67, 34, 89, 5]. "
    "Find the mean, median, and mode of these numbers."
)

print("=" * 50)
print("问题: Find mean, median, and mode of [23, 45, 12, 67, 34, 89, 5]")
print("=" * 50)
print("\n最终答案:")
print(result["response"].choices[0].message.content)

问题: Find mean, median, and mode of [23, 45, 12, 67, 34, 89, 5]

最终答案:
根据您提供的数字列表 [23, 45, 12, 67, 34, 89, 5]，统计结果如下：

🔢 **平均数（Mean）**：39.29  
（计算方式：所有数字之和 ÷ 数字个数）

📍 **中位数（Median）**：34  
（将数字从小到大排序后，中间位置的数）

🎯 **众数（Mode）**：无唯一众数  
（每个数字均只出现1次，没有重复值）

需要进一步分析或可视化这些结果吗？ 😊


## 10. 总结

### 10.1 关键要点

- **代码执行是使智能体式应用变得极其强大的关键工具**。
- **行业趋势**：许多大型语言模型的训练者会专门优化模型，以确保其在应用中代码执行功能的良好表现。
- **安全性**：必须在沙盒环境中执行代码，避免安全风险。
- **反思模式**：通过迭代反馈，模型可以自我修正错误，提高成功率。

### 10.2 未来方向

- 开发者需要手动为智能体系统创建并添加工具。
- 一个新的标准——**MCP (Model Context Protocol)** 正在兴起，它旨在让开发者更容易地访问一套庞大的工具集，供大型语言模型使用，从而简化开发流程。

### 10.3 什么时候使用代码执行？

**适合使用代码执行的场景**：
- 数学计算和科学问题
- 数据分析和处理
- 算法实现
- 需要灵活性的场景
- 不确定用户会提出什么样的问题

**更适合使用预定义工具的场景**：
- 操作外部系统（如数据库、API）
- 需要严格控制安全性的场景
- 操作频繁且性能敏感的场景
- 需要与特定硬件或服务交互的场景

<div style="border:1px solid #22c55e; border-left:6px solid #16a34a; background:#dcfce7; border-radius:6px; padding:14px 16px; color:#064e3b; font-family:system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,Cantarell,Noto Sans,sans-serif;">

🎉 <strong>恭喜！</strong>

你刚刚探索了<strong>代码执行（Code Execution）</strong>这一强大的智能体能力。你了解了：

<ul>
  <li><strong>代码执行的优势</strong>：相比于预定义工具，让模型自己写代码提供了极大的灵活性和可扩展性，能处理各种复杂的计算任务。</li>
  <li><strong>实现方法</strong>：通过提示词设计、代码提取、代码执行和结果格式化，构建了一个完整的代码执行智能体系统。</li>
  <li><strong>反思模式</strong>：当代码执行失败时，将错误信息反馈给模型，让它自我修正并重新生成代码，提高了成功率。</li>
  <li><strong>安全考量</strong>：了解了代码执行的安全风险，以及使用沙盒环境（如 Docker、E2B）的重要性。</li>
  <li><strong>应用场景</strong>：代码执行特别适合数学计算、数据分析、算法实现等需要灵活性的场景。</li>
</ul>

通过代码执行，智能体的能力边界得到了极大的扩展。你不再需要为每个可能的功能预先创建工具，而是让模型自己编写代码来解决问题。这正是<strong>Agentic AI</strong>的核心优势之一——<strong>自主性和适应性</strong>。

掌握代码执行后，你已准备好构建更强大的智能体系统，它们能够自主思考、编写代码、执行任务并自我修正，真正实现高度自治的 AI 智能体！🌟
</div>

## 11. 附录：完整工具函数参考

以下是本次实验中使用的所有函数：

In [ ]:
print("=" * 60)
print("代码执行智能体 - 函数参考")
print("=" * 60)

print("\n1. build_code_prompt(request_: str) -> str")
print("   构建 code execution 智能体的提示词")

print("\n2. extract_python_code(text: str) -> Optional[str]")
print("   从文本中提取被 <execute_python> 标签包裹的代码")

print("\n3. execute_python_code(code: str) -> Dict[str, Any]")
print("   安全地执行 Python 代码并返回输出和错误信息")

print("\n4. code_execution_agent(prompt: str, max_iterations: int = 3)")
print("   完整的代码执行智能体，支持反思模式")

print("\n" + "=" * 60)

## 12. OpenAI Tools 格式：get_current_time 和 get_weather_from_ip

本节将 `get_current_time` 和 `get_weather_from_ip` 函数转换为 OpenAI API 可调用的 tools 格式。

### 12.1 定义工具函数

In [ ]:
from datetime import datetime
import json

def get_current_time(timezone: str = "Asia/Shanghai"):
    """获取指定时区的当前时间
    
    Args:
        timezone: 时区名称，默认为 "Asia/Shanghai"
    
    Returns:
        包含当前时间信息的 JSON 字符串
    """
    return json.dumps({
        "timezone": timezone,
        "current_time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "timestamp": int(datetime.now().timestamp())
    }, ensure_ascii=False)

def get_weather_from_ip(ip: str = None):
    """根据 IP 地址获取天气信息（模拟）
    
    Args:
        ip: IP 地址，默认为 None（使用默认位置）
    
    Returns:
        包含天气信息的 JSON 字符串
    
    注意：这是模拟版本，实际应用中应调用真实的天气 API
    """
    # 模拟天气数据
    mock_weather_data = {
        "location": "北京",
        "temperature": 25,
        "condition": "晴天",
        "humidity": 45,
        "wind": "微风"
    }
    
    return json.dumps(mock_weather_data, ensure_ascii=False)

# 测试工具函数
print("测试 get_current_time:")
print(get_current_time())
print("\n测试 get_weather_from_ip:")
print(get_weather_from_ip())

### 12.2 OpenAI Tools 格式定义

OpenAI API 的 tools 参数需要一个列表，每个工具包含以下结构：
```json
{
  "type": "function",
  "function": {
    "name": "函数名",
    "description": "函数描述",
    "parameters": {
      "type": "object",
      "properties": {
        "参数名": {
          "type": "参数类型",
          "description": "参数描述"
        }
      },
      "required": ["必填参数列表"]
    }
  }
}
```

In [ ]:
# 定义 OpenAI tools 格式
openai_tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_time",
            "description": "获取指定时区的当前时间，默认时区为 Asia/Shanghai",
            "parameters": {
                "type": "object",
                "properties": {
                    "timezone": {
                        "type": "string",
                        "description": "时区名称，例如: Asia/Shanghai, America/New_York, Europe/London"
                    }
                },
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather_from_ip",
            "description": "根据 IP 地址获取天气信息（模拟数据）",
            "parameters": {
                "type": "object",
                "properties": {
                    "ip": {
                        "type": "string",
                        "description": "IP 地址，可选参数，不提供时返回默认位置的天气"
                    }
                },
                "required": []
            }
        }
    }
]

# 打印 tools 定义
print("OpenAI Tools 格式定义:")
print(json.dumps(openai_tools, indent=2, ensure_ascii=False))

### 12.3 工具函数映射字典

为了方便调用，创建一个函数名到函数对象的映射字典。

In [ ]:
# 工具函数映射
tools_dict = {
    "get_current_time": get_current_time,
    "get_weather_from_ip": get_weather_from_ip
}

print("工具映射字典:")
print(json.dumps({k: v.__name__ for k, v in tools_dict.items()}, indent=2, ensure_ascii=False))

### 12.4 完整的智能体集成示例

将 OpenAI tools 格式与智能体系统集成，处理工具调用。

In [ ]:
def handle_tool_calls_with_tools(response, messages, tools_dict, max_iterations=5):
    """处理工具调用
    
    Args:
        response: LLM 的响应
        messages: 对话历史
        tools_dict: 工具函数映射字典
        max_iterations: 最大迭代次数
    
    Returns:
        (final_response, messages) 元组
    """
    for iteration in range(max_iterations):
        # 检查是否有工具调用
        if tool_calls := response.choices[0].message.tool_calls:
            for tool_call in tool_calls:
                # 获取工具名称和参数
                function_name = tool_call.function.name
                function_args_str = tool_call.function.arguments
                
                # 解析参数
                try:
                    if function_args_str:
                        function_args = json.loads(function_args_str)
                    else:
                        function_args = {}
                except json.JSONDecodeError:
                    function_args = {}
                
                # 特殊处理：get_current_time 如果没有参数，使用默认时区
                if function_name == 'get_current_time' and not function_args:
                    function_args = {'timezone': 'Asia/Shanghai'}
                
                # 调用工具函数
                tool_func = tools_dict.get(function_name)
                if tool_func:
                    try:
                        tool_result = tool_func(**function_args)
                        if not isinstance(tool_result, str):
                            tool_result = json.dumps(tool_result)
                    except Exception as e:
                        tool_result = json.dumps({"error": str(e)})
                else:
                    tool_result = json.dumps({"error": f"Tool '{function_name}' not found"})
                
                # 将工具结果添加到消息历史
                messages.append({"role": "tool", "name": function_name, "content": tool_result})
            
            # 继续对话
            response = client.chat.completions.create(model=qwen_model, messages=messages, tools=openai_tools)
        else:
            # 没有工具调用，结束循环
            break
    
    # 最终调用一次以确保得到完整响应
    response = client.chat.completions.create(model=qwen_model, messages=messages, tools=openai_tools)
    return response, messages


def agent_with_tools(prompt: str, system_prompt: str = None):
    """带工具的智能体
    
    Args:
        prompt: 用户提示
        system_prompt: 系统提示（可选）
    
    Returns:
        (final_response, messages) 元组
    """
    # 构建消息历史
    messages = []
    
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    
    messages.append({"role": "user", "content": prompt})
    
    # 调用 LLM
    response = client.chat.completions.create(model=qwen_model, messages=messages, tools=openai_tools)
    
    # 处理工具调用
    final_response, messages = handle_tool_calls_with_tools(response, messages, tools_dict)
    
    return final_response, messages

### 12.5 测试智能体

In [ ]:
# 测试 1: 获取当前时间
response, messages = agent_with_tools("现在几点了？")

print("=" * 50)
print("问题: 现在几点了？")
print("=" * 50)
print("\n最终回答:")
print(response.choices[0].message.content)

# 打印对话历史
print("\n" + "=" * 50)
print("对话历史:")
print("=" * 50)
for i, msg in enumerate(messages):
    print(f"\n[{i}] {msg['role']}:")
    if 'name' in msg:
        print(f"  tool: {msg['name']}")
    print(f"  {msg['content']}")

In [ ]:
# 测试 2: 获取天气
response, messages = agent_with_tools("今天天气怎么样？")

print("=" * 50)
print("问题: 今天天气怎么样？")
print("=" * 50)
print("\n最终回答:")
print(response.choices[0].message.content)

In [ ]:
# 测试 3: 复合任务
response, messages = agent_with_tools("告诉我现在几点，还有今天的天气情况")

print("=" * 50)
print("问题: 告诉我现在几点，还有今天的天气情况")
print("=" * 50)
print("\n最终回答:")
print(response.choices[0].message.content)

### 12.6 总结

本节介绍了如何将自定义函数转换为 OpenAI API 可调用的 tools 格式：

**关键要点**：

1. **Tools 格式结构**：每个工具需要包含 `type: "function"` 和 `function` 对象

2. **函数元数据**：
   - `name`: 函数名
   - `description`: 函数描述（帮助模型理解何时调用该工具）
   - `parameters`: JSON Schema 格式的参数定义

3. **参数定义**：
   - `type`: "object"
   - `properties`: 各参数的类型和描述
   - `required`: 必填参数列表

4. **工具映射**：创建函数名到函数对象的字典，方便实际调用

5. **工具调用处理**：
   - 检测 `tool_response.tool_calls`
   - 解析工具名称和参数
   - 调用对应的函数
   - 将结果返回给 LLM 继续处理

**应用场景**：
- 获取实时信息（时间、天气、新闻）
- 查询外部系统（数据库、API）
- 执行特定操作（发送邮件、创建文件）

通过将函数包装为 OpenAI tools 格式，可以让 LLM 在需要时自动调用这些工具，极大地扩展了智能体的能力边界。